# Solar Inverter Monitoring - Google Colab Setup
This notebook demonstrates how to load simulated data, train an Anomaly Detection model (Isolation Forest), and run the FastAPI backend via `ngrok`.

In [ ]:
!pip install fastapi uvicorn pyngrok pandas scikit-learn sqlalchemy google-generativeai pydantic -q
!pip install nest-asyncio -q
import nest_asyncio
nest_asyncio.apply()

### 1. Data Preprocessing & ML Training
We simulate loading the CSV dataset and applying an Isolation Forest to detect anomalies.

In [ ]:
import pandas as pd
from sklearn.ensemble import IsolationForest
import matplotlib.pyplot as plt

# 1. Load Data (assuming you uploaded the generated CSV to Colab, or generated it locally)
try:
    df = pd.read_csv('historical_telemetry.csv')
    print("Loaded data shape:", df.shape)
    
    # 2. Preprocess & Feature Engineering
    features = ['voltage', 'current', 'temperature', 'dust_index']
    X = df[features].fillna(0)
    
    # 3. Train Anomaly Detection Model
    clf = IsolationForest(contamination=0.05, random_state=42)
    df['is_anomaly'] = clf.fit_predict(X) == -1
    
    print(f"Anomalies detected: {df['is_anomaly'].sum()}")
    
except FileNotFoundError:
    print("Please upload the historical_telemetry.csv file to Colab or run generate_mock_csv.py first.")

### 2. Start FastAPI Backend via ngrok
Ensure you have uploaded the `backend/` folder to Colab (or cloned the repo) so `main.py` is accessible.

In [ ]:
from pyngrok import ngrok
import uvicorn
from threading import Thread

# Enter your ngrok authtoken here if required
NGROK_AUTH_TOKEN = ""
if NGROK_AUTH_TOKEN:
    ngrok.set_auth_token(NGROK_AUTH_TOKEN)

# Start ngrok
public_url = ngrok.connect(8000).public_url
print("===== REPLACING API_URL IN REACT =====")
print("Update your frontend api.js API_URL to:", public_url + "/api")
print("========================================")

In [ ]:
import os
import sys

# Make sure the backend directory is in the Python path if uploaded to Colab
if os.path.isdir('backend'):
    sys.path.append(os.path.abspath('backend'))
    import main 
    
    def run_server():
        uvicorn.run(main.app, host="127.0.0.1", port=8000)
    
    server_thread = Thread(target=run_server)
    server_thread.start()
else:
    print("Please upload the backend folder to Colab first.")